# Numerical experiments: the DNN relaxation gap of the standard quadratic program

$$p^*(Q)=\min\{x^\top Qx:\ x\in\Delta\},\qquad
p_{\mathrm{DNN}}(Q)=\min\{\langle Q,X\rangle:\ X\succeq0,\ X\ge0,\ \langle J,X\rangle=1\},\qquad
R(Q)=\frac{p^*(Q)}{p_{\mathrm{DNN}}(Q)}.$$

Contents:

1. Certified primitives.
2. Exhaustive verification for circulant $Q$ of order five.
3. A conic reformulation of $\sup R$ and a two-block global search.
4. Local analysis at $Q^\dagger=\sqrt5\,(I+A_{C_5})$.
5. The finite family $Q=I+A_{\bar G}$, all graphs on five vertices.

## 0. Setup

In [ ]:
import sys, platform, itertools, time
from fractions import Fraction
import numpy as np, scipy, cvxpy as cp
from scipy.optimize import linprog

SEED  = 20260908
SQRT5 = np.sqrt(5.0)
BOUND = SQRT5/2
J5    = np.ones((5,5))

print("python %s | numpy %s | scipy %s | cvxpy %s"
      % (sys.version.split()[0], np.__version__, scipy.__version__, cp.__version__))
print("solver: CLARABEL (interior point)")
print("sqrt(5)/2 = %.16f" % BOUND)

## 1. Certified primitives

### 1.1 $p^*$

$p^*$ is computed by support enumeration. For a global minimiser with inclusion-minimal
support the KKT system is nonsingular, so the singular branch may be skipped. Every
candidate evaluated is $x^\top Qx$ at a feasible $x$, hence an upper bound for $p^*$.

In [ ]:
def p_star(Q, tol=1e-11):
    Q = np.asarray(Q, float); n = Q.shape[0]
    best, arg = np.inf, None
    for r in range(1, n+1):
        for S in itertools.combinations(range(n), r):
            S = list(S); k = r
            A = np.zeros((k+1, k+1)); b = np.zeros(k+1)
            A[:k,:k] = Q[np.ix_(S,S)]; A[:k,k] = -1.0; A[k,:k] = 1.0; b[k] = 1.0
            try: sol = np.linalg.solve(A, b)
            except np.linalg.LinAlgError: continue
            xs = sol[:k]
            if np.any(xs < -tol): continue
            x = np.zeros(n); x[S] = np.maximum(xs, 0.0)
            s = x.sum()
            if s <= 0: continue
            x /= s
            val = float(x @ Q @ x)
            if val < best: best, arg = val, x
    return best, arg


def p_star_rational(Q):
    n = len(Q); Q = [[Fraction(Q[i][j]) for j in range(n)] for i in range(n)]
    best = arg = None
    for r in range(1, n+1):
        for S in itertools.combinations(range(n), r):
            k = r; M = [[Fraction(0)]*(k+2) for _ in range(k+1)]
            for a,i in enumerate(S):
                for bb,j in enumerate(S): M[a][bb] = Q[i][j]
                M[a][k] = Fraction(-1)
            for bb in range(k): M[k][bb] = Fraction(1)
            M[k][k+1] = Fraction(1)
            row, sing = 0, False
            for col in range(k+1):
                p = next((rr for rr in range(row,k+1) if M[rr][col] != 0), None)
                if p is None: sing = True; break
                M[row], M[p] = M[p], M[row]
                pv = M[row][col]; M[row] = [v/pv for v in M[row]]
                for rr in range(k+1):
                    if rr != row and M[rr][col] != 0:
                        f = M[rr][col]
                        M[rr] = [a2-f*b2 for a2,b2 in zip(M[rr], M[row])]
                row += 1
            if sing or row != k+1: continue
            xs = [M[i][k+1] for i in range(k+1)][:k]
            if any(v < 0 for v in xs): continue
            x = [Fraction(0)]*n
            for a,i in enumerate(S): x[i] = xs[a]
            tot = sum(x)
            if tot <= 0: continue
            x = [v/tot for v in x]
            val = sum(Q[i][j]*x[i]*x[j] for i in range(n) for j in range(n))
            if best is None or val < best: best, arg = val, tuple(x)
    return best, arg

### 1.2 $p_{\mathrm{DNN}}$ with rigorous bounds

Dual: $\max\{\lambda:\ Q-\lambda J=S+N,\ S\succeq0,\ N\ge0\}$. Solver tolerances are not
used as accuracy claims. Instead the returned primal and dual points are rounded to exactly
feasible ones, giving a two-sided interval whose width is the attained duality gap.

In [ ]:
def p_dnn(Q, ret_X=False):
    n = Q.shape[0]; X = cp.Variable((n,n), symmetric=True)
    pr = cp.Problem(cp.Minimize(cp.sum(cp.multiply(Q, X))),
                    [X >> 0, X >= 0, cp.sum(X) == 1])
    pr.solve(solver=cp.CLARABEL)
    return (pr.value, X.value) if ret_X else pr.value


def dnn_upper(Q):
    _, Xv = p_dnn(Q, ret_X=True)
    X = (Xv + Xv.T)/2
    w, V = np.linalg.eigh(X); X = V @ np.diag(np.maximum(w, 0)) @ V.T
    X = np.maximum((X + X.T)/2, 0.0)
    # clipping entries can destroy psd. I is psd AND entrywise nonnegative, so
    # adding a multiple of it restores psd without leaving the nonnegative orthant
    t = max(0.0, -float(np.linalg.eigvalsh(X).min()))
    X = X + t*np.eye(Q.shape[0]); X /= X.sum()
    return float(np.sum(Q*X)), X


def dnn_lower(Q):
    n = Q.shape[0]; Jn = np.ones((n,n))
    lam = cp.Variable(); S = cp.Variable((n,n), symmetric=True); N = cp.Variable((n,n))
    cp.Problem(cp.Maximize(lam),
               [Q - lam*Jn == S + N, S >> 0, N >= 0]).solve(solver=cp.CLARABEL)
    lam0 = float(lam.value); S0 = np.array(S.value); S0 = (S0 + S0.T)/2
    w, V = np.linalg.eigh(S0); Sp = (V @ np.diag(np.maximum(w, 0)) @ V.T)
    Sp = (Sp + Sp.T)/2
    lam_c = lam0 - max(0.0, float(np.max(-(Q - lam0*Jn - Sp))))
    Nc = Q - lam_c*Jn - Sp
    ok = (Nc.min() >= -1e-14) and (np.linalg.eigvalsh(Sp).min() >= -1e-14)
    return lam_c, ok, (float(Nc.min()), float(np.linalg.eigvalsh(Sp).min()))


def ratio(Q):
    return p_star(Q)[0]/p_dnn(Q)


def circ(d, e, f):
    Q = np.zeros((5,5))
    for i in range(5):
        Q[i,i] = d
        Q[i,(i+1)%5] = Q[(i+1)%5,i] = e
        Q[i,(i+2)%5] = Q[(i+2)%5,i] = f
    return Q


def p_dnn_circ(d, e, f):
    return min(d, SQRT5/5*d + (1-SQRT5/5)*f,
               (d + 2*e + 2*f)/5, SQRT5/5*d + (1-SQRT5/5)*e)

### 1.3 Validation

In [ ]:
A_C5 = np.zeros((5,5))
for i in range(5): A_C5[i,(i+1)%5] = A_C5[(i+1)%5,i] = 1.0
Qdag = SQRT5*(np.eye(5) + A_C5)

ps, xs = p_star(Qdag)
lo, ok, res = dnn_lower(Qdag); up, Xup = dnn_upper(Qdag)
print("Q = sqrt(5) (I + A_C5)")
print("  p*        = %.15f   (sqrt5/2 = %.15f)" % (ps, BOUND))
print("  minimiser =", np.round(xs, 12))
print("  p_DNN in [%.15f, %.15f], gap %.2e, dual feasible %s" % (lo, up, up-lo, ok))
print("     dual residuals   : min N = %.1e, min eig S = %.1e" % res)
print("     primal residuals : min entry = %.1e, min eig = %.1e"
      % (Xup.min(), np.linalg.eigvalsh(Xup).min()))
print("  R     in [%.15f, %.15f]" % (ps/up, ps/lo))

In [ ]:
rng = np.random.default_rng(SEED); err = gap = 0.0; outside = 0; worst = 0.0
for _ in range(200):
    d, e, f = rng.uniform(0, 5, 3); Q = circ(d, e, f)
    v = p_dnn(Q); lo = dnn_lower(Q)[0]; up = dnn_upper(Q)[0]
    err = max(err, abs(p_dnn_circ(d,e,f) - v)); gap = max(gap, up - lo)
    if v < lo: outside += 1; worst = min(worst, v - lo)
print("circulant closed form vs SDP, 200 instances: max deviation %.1e" % err)
print("worst certified duality gap on the same set : %.1e" % gap)
print("raw solver value below the certified lower bound: %d / 200 (worst %.1e)" % (outside, worst))
print("  the dual triple is exactly feasible, so p_DNN >= lo holds; it is the solver")
print("  value that is off. A solver tolerance is not an accuracy certificate.")

In [ ]:
Qint = [[1 if i == j else (1 if (i-j) % 5 in (1,4) else 0) for j in range(5)] for i in range(5)]
val, xr = p_star_rational(Qint)
print("exact p*(I + A_C5) =", val, " | float %.17f" % p_star(np.array(Qint, float))[0])
print("exact minimiser    =", xr)

## 2. Circulant $Q$ of order five

Write $Q=\mathrm{circ}(d,e,f)$  $d$ on the diagonal, $e$ on $\{i,i\!+\!1\}$, $f$ on $\{i,i\!+\!2\}$.

**Reductions.** $R$ is scale invariant and $p_{\mathrm{DNN}}>0$ forces $d>0$, so set $d=1$.
The relabelling $i\mapsto 2i \bmod 5$ gives $R(1,e,f)=R(1,f,e)$. If $e,f\ge1$ then $Q\ge J$,
whence $p^*=p_{\mathrm{DNN}}=1$. If $e\ge1\ge f$ then $p^*=\min\{1,(1+f)/2\}$ is independent of
$e$ while $p_{\mathrm{DNN}}$ is nondecreasing in $e$, so $R$ is non-increasing in $e$ and the
strip is dominated by $e=1$. Hence $\sup R$ over the cone equals $\max R$ over $[0,1]^2$.

In [ ]:
rng = np.random.default_rng(SEED+2); bad3 = bad4 = 0
for _ in range(300):
    e, f = rng.uniform(1, 40, 2)
    if abs(p_star(circ(1,e,f))[0]-1) > 1e-9 or abs(p_dnn_circ(1,e,f)-1) > 1e-9: bad3 += 1
for _ in range(300):
    f = rng.uniform(0,1); e = rng.uniform(1,60)
    if abs(p_star(circ(1,e,f))[0] - min(1.0,(1+f)/2)) > 1e-9: bad4 += 1
    if (p_star(circ(1,e,f))[0]/p_dnn_circ(1,e,f)
        > p_star(circ(1,1,f))[0]/p_dnn_circ(1,1,f) + 1e-9): bad4 += 1
print("corner e,f >= 1 gives R = 1 : %d exceptions / 300" % bad3)
print("strip dominated by e = 1    : %d exceptions / 600" % bad4)

**Box bound.** With $\theta=(1,e,f)$ and
$s(x)=\big(\sum x_i^2,\ 2\sum x_ix_{i+1},\ 2\sum x_ix_{i+2}\big)$ one has
$x^\top Qx=\langle\theta,s(x)\rangle$. For any feasible $x_c$, $\theta\mapsto\langle\theta,s(x_c)\rangle$
is an affine majorant of $p^*$ valid on the whole box, maximised at a corner.
$p_{\mathrm{DNN}}$ is concave in $\theta$, so its minimum on a box is at a corner. Dividing
gives a rigorous upper bound for $R$ on the box. Subdivide until every box clears the target.

In [ ]:
def moments(x):
    return np.array([float(x @ x),
                     2.0*sum(x[i]*x[(i+1)%5] for i in range(5)),
                     2.0*sum(x[i]*x[(i+2)%5] for i in range(5))])

def bnb(tol, target=BOUND):
    cache = {}
    def P(e, f):
        k = (round(e,15), round(f,15))
        if k not in cache: cache[k] = p_dnn_circ(1.0, e, f)
        return cache[k]
    stack = [(0.,1.,0.,1.)]; nb = leaves = 0; worst = 0.0; arg = None
    while stack:
        e0, e1, f0, f1 = stack.pop(); nb += 1
        cor = [(e0,f0),(e0,f1),(e1,f0),(e1,f1)]
        lo = min(P(e,f) for e,f in cor)
        ec, fc = 0.5*(e0+e1), 0.5*(f0+f1)
        s = moments(np.asarray(p_star(circ(1.0,ec,fc))[1], float))
        hi = max(s[0] + e*s[1] + f*s[2] for e,f in cor)
        if lo > 0 and hi/lo <= target + tol:
            leaves += 1
            r = p_star(circ(1.0,ec,fc))[0]/P(ec,fc)
            if r > worst: worst, arg = r, (ec,fc)
            continue
        if (e1-e0) >= (f1-f0): stack += [(e0,ec,f0,f1),(ec,e1,f0,f1)]
        else:                  stack += [(e0,e1,f0,fc),(e0,e1,fc,f1)]
    return nb, leaves, target+tol, worst, arg

print("%8s %8s %8s %20s %8s" % ("tol","boxes","leaves","certified R <=","time"))
for tol in (1e-3, 1e-5, 1e-7, 1e-8):
    t0 = time.time(); nb, lv, cb, w, a = bnb(tol); el = time.time()-t0
    print("%8.0e %8d %8d %20.10f %7.1fs" % (tol, nb, lv, cb, el))
print("\nsqrt(5)/2 = %.10f" % BOUND)
print("largest ratio met in the final subdivision: %.10f at (e,f) = (%.6f, %.6f)" % (w, a[0], a[1]))

The subdivision terminates, so $R\le\sqrt5/2+10^{-8}$ over all nonnegative symmetric
circulant $Q$ of order five. The bound is approached as $f\to0$, $e\to1$.

**The equality family.** $R=\sqrt5/2$ exactly on $f=0,\ e\ge d>0$.

In [ ]:
print("f = 0, e >= d")
for d, e in [(1,1),(1,2),(1,10),(0.3,0.31),(2.5,1e6),(1e-6,1e-6)]:
    r = p_star(circ(d,e,0.0))[0]/p_dnn_circ(d,e,0.0)
    print("   d=%-8g e=%-8g R = %.15f   R - sqrt5/2 = %+.1e" % (d, e, r, r-BOUND))
print("e < d")
for d, e in [(1,0.999),(1,0.99),(1,0.9),(1,0.5),(1,0.0)]:
    print("   d=%-6g e=%-6g R = %.15f" % (d, e, p_star(circ(d,e,0.0))[0]/p_dnn_circ(d,e,0.0)))

## 3. Conic reformulation and two-block global search

Let $\mathcal D=\{X\succeq0,\ X\ge0,\ \langle J,X\rangle=1\}$. Using
$p^*(Q)\ge t\iff Q-tJ\in\mathcal{COP}^5$, weak duality and homogeneity, substituting
$Q=t(J+U)$ gives

$$\rho_5=\sup_{Q\ge0}R(Q)=\frac{1}{1+\Psi^*},\qquad
\Psi^*=\min\{\langle X,U\rangle:\ X\in\mathcal D,\ U\in\mathcal{COP}^5,\ U\ge-J\},$$

and the conjecture $\rho_5=\sqrt5/2$ reads $\Psi^*=2/\sqrt5-1$.

With $\Psi(X)=\min_U\langle X,U\rangle$ and $g(X)=1/(1+\Psi(X))$: $\Psi$ is concave, so $g$ is
convex and its maximum over $\mathcal D$ is attained at an extreme point. Moreover
$g(X)>1$ iff $X\notin\mathcal{CP}^5$. By Burer, Anstreicher and Dür (2009) an extreme
$X\in\mathcal{DNN}^5\setminus\mathcal{CP}^5$ has a $5$-cycle zero pattern and rank $3$, so
after relabelling it lies in the exposed face
$\mathcal F=\{X\in\mathcal D:\ X_{i,i+1}=0\}$. Rank-3 elements of $\mathcal F$ are Gram
matrices of five unit vectors in $\mathbb R^3$ with $v_i\perp v_{i+1}$ cyclically: two shape
parameters, five diagonal scalings, one normalisation.

In [ ]:
CYC = [(i,(i+1)%5) for i in range(5)]

def frame(a, b):
    v1 = np.array([1.,0.,0.]); v2 = np.array([0.,1.,0.])
    v3 = np.array([np.cos(a), 0., np.sin(a)])
    u1 = np.array([-np.sin(a), 0., np.cos(a)])
    v4 = np.cos(b)*u1 + np.sin(b)*v2
    v5 = np.array([0., np.cos(b)*np.cos(a), -np.sin(b)])
    n = np.linalg.norm(v5)
    return None if n < 1e-12 else np.vstack([v1, v2, v3, v4, v5/n])

def extreme_dnn(a, b, diag=None):
    V = frame(a, b)
    if V is None: return None
    for m in range(32):
        s = np.array([1. if (m >> k) & 1 == 0 else -1. for k in range(5)])
        Y = (s[:,None]*V) @ (s[:,None]*V).T
        if Y.min() > -1e-11: break
    else: return None
    Y = np.maximum(Y, 0.0)
    for i,j in CYC: Y[i,j] = Y[j,i] = 0.0
    if diag is not None:
        D = np.diag(np.asarray(diag, float)); Y = D @ Y @ D
    s = Y.sum()
    return Y/s if s > 1e-12 else None

rng = np.random.default_rng(SEED); ranks = []
for _ in range(300):
    X = extreme_dnn(rng.uniform(.02, np.pi-.02), rng.uniform(.02, np.pi-.02),
                    np.exp(rng.normal(0, .6, 5)))
    if X is not None: ranks.append(np.linalg.matrix_rank(X, tol=1e-9))
print("sampler: %d/300 valid, ranks %s" % (len(ranks), dict(zip(*np.unique(ranks, return_counts=True)))))

$\langle X,U\rangle$ is bilinear over two convex sets. The $U$-step minimises over
$\{U\in\mathcal{COP}^5,\ U\ge-J\}$ by cutting planes. Each cut $x^\top Ux\ge0$ is valid for
every copositive $U$, so the pool is shared across objectives, and separation is exact via
$p^*$. The $X$-step is an SDP over $\mathcal F$, an exposed face of $\mathcal D$, so iterates
are extreme rays of $\mathcal{DNN}^5$. Both half-steps are solved globally. LP values are
lower bounds, so each limit is re-evaluated at a repaired, certified copositive $U$.

In [ ]:
IDX = [(i,j) for i in range(5) for j in range(i+1,5)]
def cut_row(x): return np.array([x[i]**2 for i in range(5)] + [2*x[i]*x[j] for i,j in IDX])
def unvec(v):
    U = np.zeros((5,5))
    for i in range(5): U[i,i] = v[i]
    for k,(i,j) in enumerate(IDX): U[i,j] = U[j,i] = v[5+k]
    return U

class CopModel:
    def __init__(self, cap=50.0, maxcuts=400):
        self.rows  = [cut_row(np.eye(5)[i]) for i in range(5)]
        self.rows += [cut_row((np.eye(5)[i]+np.eye(5)[j])/2) for i,j in IDX]
        self.rows += [cut_row(np.ones(5)/5)]
        self.cap, self.maxcuts, self.nsep = cap, maxcuts, 0
    def solve(self, C, tol=1e-9, max_rounds=60):
        c = np.array([C[i,i] for i in range(5)] + [2*C[i,j] for i,j in IDX])
        bounds = [(0., self.cap)]*5 + [(-1., self.cap)]*10
        for _ in range(max_rounds):
            res = linprog(c, A_ub=-np.array(self.rows), b_ub=np.zeros(len(self.rows)),
                          bounds=bounds, method="highs")
            if not res.success: return np.nan, None
            U = unvec(res.x); viol, xv = p_star(U); self.nsep += 1
            if viol >= -tol: return float(res.fun), U
            self.rows.append(cut_row(xv))
            if len(self.rows) > self.maxcuts:
                self.rows = self.rows[:16] + self.rows[-(self.maxcuts-16):]
        return float(res.fun), U

class DnnFace:
    def __init__(self):
        self.U = cp.Parameter((5,5), symmetric=True)
        self.X = cp.Variable((5,5), symmetric=True)
        cons = [self.X >> 0, self.X >= 0, cp.sum(self.X) == 1]
        cons += [self.X[i,j] == 0 for i,j in CYC]
        self.pr = cp.Problem(cp.Minimize(cp.sum(cp.multiply(self.U, self.X))), cons)
    def __call__(self, U):
        self.U.value = (U+U.T)/2; self.pr.solve(solver=cp.CLARABEL)
        Xv = np.array(self.X.value); X = np.maximum((Xv+Xv.T)/2, 0.0)
        return float(self.pr.value), X/X.sum()

def certify(X, U):
    '''Value of <X,U> at an exactly feasible pair, hence an upper bound for Psi*.'''
    if U is None or X is None: return np.nan, None
    # U: raising entries preserves copositivity, so clip to U >= -J first. The cap can
    # break it, so repair AFTER clipping by adding a multiple of I (sum x_i^2 >= 1/5)
    U = np.minimum(np.maximum(np.asarray(U, float), -1.0), 50.0)
    np.fill_diagonal(U, np.maximum(np.diag(U), 0.))
    viol, _ = p_star(U)
    if viol < 0: U = U + 5.0*abs(viol)*np.eye(5)
    # X: it lies in the face F, so set its cycle entries exactly to zero, then restore
    # psd with a multiple of I (psd and entrywise nonnegative) and renormalise
    X = np.maximum(np.asarray(X, float), 0.0)
    for i, j in CYC: X[i,j] = X[j,i] = 0.0
    t = max(0.0, -float(np.linalg.eigvalsh(X).min()))
    X = (X + t*np.eye(5)); X = X/X.sum()
    return float(np.sum(X*U)), U

TARGET = 2.0/SQRT5 - 1.0
print("Psi* target = 2/sqrt(5) - 1 = %.15f" % TARGET)

In [ ]:
M, DF = CopModel(), DnnFace()

a, c = SQRT5/25, (5-SQRT5)/50
Xdag = np.zeros((5,5))
for i in range(5):
    Xdag[i,i] = a; Xdag[i,(i+2)%5] = Xdag[(i+2)%5,i] = c
Xdag /= Xdag.sum()

v_dag, U_dag = M.solve(Xdag)
print("Psi at the circulant extreme ray = %.12f   (target %.12f)" % (v_dag, TARGET))
print("Psi at J/25 (completely positive) = %.12f" % M.solve(J5/25)[0])
print("minimising U, rounded:\n", np.round(U_dag, 4))
print("diagonal 1, chords -1; the cycle entries are unconstrained because X vanishes there.")

In [ ]:
t0 = time.time(); rng = np.random.default_rng(SEED); vals = []; best = (np.inf, None, None)
for k in range(40):
    X = None
    for _ in range(50):
        X = extreme_dnn(rng.uniform(.02, np.pi-.02), rng.uniform(.02, np.pi-.02),
                        np.exp(rng.normal(0, .6, 5)))
        if X is not None and X.min() >= 0: break
    if X is None: continue
    U = None; prev = None
    for _ in range(25):
        _, Un = M.solve(X)
        if Un is None: break
        U = Un; v, X = DF(U)
        if prev is not None and abs(prev-v) < 1e-12: break
        prev = v
    vc, Uc = certify(X, U)
    if np.isfinite(vc):
        vals.append(vc)
        if vc < best[0]: best = (vc, X.copy(), Uc.copy())
vals = np.array(vals)

print("%d starts, %.0fs, %d cuts pooled, %d exact separations"
      % (len(vals), time.time()-t0, len(M.rows), M.nsep))
print("  best certified Psi   = %.12f   (target %.12f, gap %+.1e)"
      % (vals.min(), TARGET, vals.min()-TARGET))
print("  largest certified    = %.12f                    (gap %+.1e)"
      % (vals.max(), vals.max()-TARGET))
print("  starts reaching it   = %d / %d" % (int((vals < TARGET+1e-8).sum()), len(vals)))
print("  starts below target  = %d" % int((vals < TARGET-1e-8).sum()))
print("  implied sup R        = %.12f   (sqrt5/2 = %.12f)" % (1/(1+vals.min()), BOUND))
print("  distinct limits      :", [(float(x), int(y))
                                   for x, y in zip(*np.unique(np.round(vals,8), return_counts=True))])

In [ ]:
vbest, Xb, Ub = best
print("optimal X, normalised by its largest entry:")
print(np.round(Xb/Xb.max(), 6))
print("off-diagonal entry %.9f   ((sqrt5-1)/2 = %.9f),  rank %d"
      % (Xb[0,2]/Xb[0,0], (SQRT5-1)/2, np.linalg.matrix_rank(Xb, tol=1e-9)))
Qr = J5 + Ub
ps, _ = p_star(Qr); lo,_,_ = dnn_lower(Qr); up,_ = dnn_upper(Qr)
print("\nQ = J + U is circulant with d = %.4f, e = %.4f, f = %.4f" % (Qr[0,0], Qr[0,1], Qr[0,2]))
print("R in [%.10f, %.10f]" % (ps/up, ps/lo))

## 4. Local analysis at $Q^\dagger=\sqrt5\,(I+A_{C_5})$

Let $\rho=\sqrt5/2$, let $\mathcal M$ be the set of global minimisers of the StQP at
$Q^\dagger$ with $P_k=x_kx_k^\top$, and let $X^\dagger$ be the (unique) DNN optimiser. For a
direction $D$, Danskin's theorem gives
$R'(D)=\min_k\langle D,P_k\rangle-\rho\langle D,X^\dagger\rangle$. $D$ is admissible if
$D\ge0$ wherever $Q^\dagger_{ij}=0$. With $C_\theta=\frac15\sum_kP_k$,
$R'(D)\le\langle D,\ C_\theta-\rho X^\dagger\rangle$, so $Q^\dagger$ is a local maximiser once

$$C_\theta-\rho X^\dagger=0\ \text{ on }\mathrm{supp}(Q^\dagger),\qquad
C_\theta-\rho X^\dagger\le0\ \text{ off it.}$$

In [ ]:
supp = (Qdag > 1e-12)
ps_d, _ = p_star(Qdag); pd_d, Xv = p_dnn(Qdag, ret_X=True)
Xd = np.maximum((Xv+Xv.T)/2, 0.0); Xd /= Xd.sum()

pts = []
for r in range(1,6):
    for S in itertools.combinations(range(5), r):
        QS = Qdag[np.ix_(S,S)]
        if abs(np.linalg.det(QS)) < 1e-10: continue
        w = np.linalg.solve(QS, np.ones(r))
        if w.min() < -1e-12 or w.sum() <= 1e-12: continue
        x = np.zeros(5); x[list(S)] = np.maximum(w,0)/w.sum()
        if float(x @ Qdag @ x) < ps_d + 1e-9 and not any(np.allclose(x,y,atol=1e-8) for y in pts):
            pts.append(x)
P = [np.outer(x,x) for x in pts]
print("global minimisers: %d distinct points" % len(pts))
for x in pts: print("   ", np.round(x, 6))

In [ ]:
# uniqueness of the DNN optimiser, exactly as in the proof. Complementarity with the
# dual triple forces X_{i,i+1} = 0 and range(X) in ker S, and the five resulting
# conditions have rank five on the six-dimensional space {B G B^T}
nu = (3*SQRT5 - 5)/2
Ndual = nu*A_C5; S = Qdag - J5 - Ndual
print("dual triple (1, S, N): N >= 0 %s, residual %.1e" %
      (Ndual.min() >= 0, np.abs(Qdag - J5 - S - Ndual).max()))
print("eig(S) =", np.round(np.linalg.eigvalsh(S), 12), " -> rank", int((np.linalg.eigvalsh(S) > 1e-9).sum()))
B = np.column_stack([np.ones(5),
                     [np.cos(4*np.pi*j/5) for j in range(5)],
                     [np.sin(4*np.pi*j/5) for j in range(5)]])
print("max |S B| = %.1e   (B spans ker S)" % np.abs(S @ B).max())
bas = []
for p in range(3):
    for q in range(p, 3):
        E = np.zeros((3,3)); E[p,q] = E[q,p] = 1.0
        bas.append(B @ E @ B.T)
M5 = np.array([[Y[i,(i+1)%5] for Y in bas] for i in range(5)])
print("rank of the 5x6 system X_{i,i+1} = 0 : %d   (5 => solution set is a line)"
      % np.linalg.matrix_rank(M5, tol=1e-9))

Ct = sum(P)/len(P); Gap = Ct - BOUND*Xd
print("\nX_dagger  diag %.10f   cycle %.1e   chord %.10f" % (Xd[0,0], abs(Xd[0,1]), Xd[0,2]))
print("predicted      %.10f         0          %.10f" % (SQRT5/25, (5-SQRT5)/50))
print("\nC_theta - rho X_dagger:  on supp  max |.| = %.1e   (must be 0)" % np.abs(Gap[supp]).max())
print("                         off supp max     = %+.10f  (must be <= 0)" % Gap[~supp].max())
print("slack per chord entry = %.12f ;  (sqrt5-2)/20 = %.12f" % (-Gap[0,2], (SQRT5-2)/20))

In [ ]:
R0 = ps_d/pd_d
def admissible(rng, circulant):
    D = circ(*rng.normal(size=3)) if circulant else None
    if D is None:
        G = rng.normal(size=(5,5)); D = (G+G.T)/2
    D[~supp] = np.abs(D[~supp])
    return D/np.abs(D).max()

rng = np.random.default_rng(SEED+3)
print("%-22s %14s %14s %16s" % ("direction","R' upper bnd","FD t = 1e-5","max_t R(Q+tD)-R0"))
for tag, cf in [("circulant",True),("circulant",True),
                ("symmetry-breaking",False),("symmetry-breaking",False),
                ("symmetry-breaking",False)]:
    D = admissible(rng, cf)
    pred = min(float(np.sum(D*Pk)) for Pk in P) - BOUND*float(np.sum(D*Xd))
    fd = (ratio(Qdag + 1e-5*D) - R0)/1e-5
    mx = max(ratio(Qdag + t*D) - R0 for t in (1e-1,1e-2,1e-3,1e-4,1e-5,1e-6))
    print("%-22s %14.6f %14.6f %16.1e" % (tag, pred, fd, mx))

rng = np.random.default_rng(SEED+4); preds = []; asc = 0
for k in range(400):
    D = admissible(rng, k % 3 == 0)
    preds.append(min(float(np.sum(D*Pk)) for Pk in P) - BOUND*float(np.sum(D*Xd)))
    if max(ratio(Qdag + t*D) for t in (1e-2,1e-3,1e-4,1e-5)) > R0 + 1e-10: asc += 1
print("\n400 admissible directions (one third circulant):")
print("   largest R' upper bound = %+.8f" % max(preds))
print("   ascent directions      = %d" % asc)

## 5. The family $Q=I+A_{\bar G}$

Here $p^*=1/\alpha(G)$ and $p_{\mathrm{DNN}}=1/\vartheta'(G)$, so $R=\vartheta'(G)/\alpha(G)$.
On five vertices there are 34 graphs up to isomorphism, so the family is checked exhaustively.

In [ ]:
pairs = list(itertools.combinations(range(5), 2))
pidx  = {p: k for k, p in enumerate(pairs)}
perms = list(itertools.permutations(range(5)))

def canon(mask):
    best = None
    for p in perms:
        m = 0
        for k,(i,j) in enumerate(pairs):
            if mask >> k & 1: m |= 1 << pidx[tuple(sorted((p[i], p[j])))]
        best = m if best is None else min(best, m)
    return best

def alpha(A):
    return max(r for r in range(6)
               for S in itertools.combinations(range(5), r)
               if all(A[i,j] == 0 for i,j in itertools.combinations(S,2)))

def theta_prime(A):
    X = cp.Variable((5,5), symmetric=True)
    cons = [X >> 0, X >= 0, cp.trace(X) == 1]
    cons += [X[i,j] == 0 for i in range(5) for j in range(i+1,5) if A[i,j]]
    pr = cp.Problem(cp.Maximize(cp.sum(X)), cons); pr.solve(solver=cp.CLARABEL)
    return float(pr.value)

seen, rows = set(), []
for mask in range(1024):
    cmask = canon(mask)
    if cmask in seen: continue
    seen.add(cmask)
    A = np.zeros((5,5))
    for k,(i,j) in enumerate(pairs):
        if mask >> k & 1: A[i,j] = A[j,i] = 1
    Q = np.eye(5) + (1 - np.eye(5) - A)
    ps, _ = p_star(Q); pd = p_dnn(Q)
    rows.append((int(A.sum())//2, alpha(A), theta_prime(A), ps, pd, ps/pd, A.copy()))
rows.sort(key=lambda t: -t[5])

print("%d isomorphism classes\n" % len(rows))
print("%6s %6s %10s %10s %10s %14s" % ("edges","alpha","theta'","p*","p_DNN","R"))
for e, al, th, ps, pd, r, _ in rows[:6]:
    print("%6d %6d %10.6f %10.6f %10.6f %14.10f" % (e, al, th, ps, pd, r))
print("\nmaximum R = %.12f   (sqrt5/2 = %.12f)" % (rows[0][5], BOUND))
print("classes with R > 1 + 1e-6 : %d" % sum(1 for r in rows if r[5] > 1+1e-6))
print("second largest R          : %.9f  (i.e. 1 to solver accuracy)" % rows[1][5])
A = rows[0][6]
print("\nmaximiser: degree sequence %s" % sorted(int(v) for v in A.sum(1)))
print(A.astype(int))